# 10 — Multi-VSL: trích xuất pose RTMPose-L WholeBody

Notebook **độc lập** này chuẩn bị dữ liệu pose cho **M-VSL200, đủ 3 góc quay center / left / right**:

1. Chọn lớp **chỉ theo số clip train của camera center** trong các file CSV chính thức (`*_1_200_center_ord1.csv`). Mặc định lấy 50 lớp, trùng đúng danh sách 50 từ của baseline RGB ở notebook 09.
2. Với `VIEW_MODE='three_view'`, đọc file ghép 3 góc chính thức (`*_1_200_three_view_ord1.csv`): mỗi dòng là một lần ký gồm 3 video center/left/right quay đồng thời. Ba video này dùng chung `instance_id`.
3. Tải **đúng các video cần dùng** từ thư mục Google Drive của tác giả vào `/content` (tạm thời, không chép video lên Drive của bạn).
4. Kiểm tra từng video (tồn tại, không rỗng, đọc được header bằng ffprobe, đủ 3 góc cho mỗi lần ký), sau đó ghi `manifest`, `labels`, `split` lên Drive.
5. Chạy **RTMDet-M** (phát hiện người) + **RTMPose-L 384×288 COCO-WholeBody** trên **mọi frame** của **từng video, từng góc**, và lưu 133 keypoint thô cho mỗi clip. Có thể resume.
6. Chuyển pose thô thành tensor đồ thị `[64, 75, 7]` cho graph encoder.

Với 50 lớp: 1.041 / 199 / 197 lần ký (train/val/test) × 3 góc = **4.311 video**. Có 59 clip train chỉ có góc center (tác giả không ghép được bộ ba) nên không nằm trong bộ 3 góc.

**Split:** giữ nguyên split chính thức, chia theo người ký (signer-disjoint): train 20 signer, validation 4, test 4. Ba góc của cùng một lần ký luôn nằm chung một split. Notebook **không** chia lại. Validation dùng để chọn checkpoint; test chỉ dùng một lần ở cuối.

**Tên từ:** metadata công khai của tác giả chỉ có nhãn số (0–199), không có chữ tiếng Việt. Danh sách lớp được in ở cell chọn lớp và lưu trong `prepared/selection.json`, với tên hiển thị `VSL_NNN` = nhãn gốc `NNN-1`.

Trước khi chạy: **Runtime → Change runtime type → T4 GPU** (hoặc L4/A100).

## Nguồn video

**Quan trọng:** [thư mục Google Drive công khai](https://drive.google.com/drive/folders/1yUU1m2hy_CjaXDDoR_6i9Y3T1XL2pD4C) trong README của tác giả chỉ chứa **1.000 video mẫu**, rải rác trên toàn bộ 1.000 từ, 30 người ký và 3 góc. Kiểm tra ngày 2026-09-25: trong đó chỉ có 52/4.311 video mà top-50 ba góc cần, và không có bộ ba center/left/right nào đủ cả ba. Bộ dữ liệu đầy đủ (khoảng 84.000 video) phải xin trực tiếp từ nhóm tác giả.

- `VIDEO_SOURCE = 'local_dir'`: dùng khi đã có bộ đầy đủ, ví dụ tác giả chia sẻ thư mục cho bạn (*Add shortcut to Drive*) hoặc bạn tự upload và giải nén. Đặt đường dẫn vào `LOCAL_SOURCE_DIR`; notebook chép từng file cần dùng vào `/content`.
- `VIDEO_SOURCE = 'drive_api'`: dùng Drive API liệt kê một thư mục Drive (mặc định là thư mục công khai ở trên; đổi `OFFICIAL_DRIVE_FOLDER_ID` nếu tác giả gửi thư mục khác), rồi chỉ tải các file cần dùng, có kiểm tra MD5.

Dù dùng cách nào, bước kiểm tra đều từ chối chạy tiếp nếu thiếu dù chỉ một video của các lớp đã chọn.

**Lưu ý:** khuôn mặt trong video đã bị tác giả làm mờ, nên 20 điểm mặt trong layout 75 khớp có độ tin cậy thấp. Pose thô vẫn giữ đủ 133 điểm, nên có thể đổi layout sau mà không cần chạy lại RTMPose.

In [ ]:
PROJECT_GIT_URL = 'https://github.com/stillthethrone/silent-signal.git'
PROJECT_GIT_REF = 'feat/multi-vsl-rtmpose-extraction'  # @param {type:'string'}
OFFICIAL_REPOSITORY = 'https://github.com/Etdihatthoc/Multi-VSL_WACV_2025.git'
OFFICIAL_DRIVE_FOLDER_ID = '1yUU1m2hy_CjaXDDoR_6i9Y3T1XL2pD4C'

# CLASS_COUNT = 0 lấy toàn bộ 199 lớp; 50 khớp với baseline RGB.
CLASS_COUNT = 50  # @param {type:'integer'}
# 'three_view' = center + left + right của cùng một lần ký; 'center' = chỉ camera chính diện.
VIEW_MODE = 'three_view'  # @param ['three_view', 'center']
# Thư mục công khai chỉ có 1.000 video mẫu; cần bộ đầy đủ từ tác giả (xem mục Nguồn video).
VIDEO_SOURCE = 'drive_api'  # @param ['drive_api', 'local_dir']
LOCAL_SOURCE_DIR = '/content/drive/MyDrive/Multi-VSL-videos'  # @param {type:'string'}
DOWNLOAD_WORKERS = 8  # @param {type:'integer'}
VALIDATION_LEVEL = 'probe'  # @param ['metadata', 'probe']

RUN_PILOT = True  # @param {type:'boolean'}
PILOT_LIMIT = 20  # @param {type:'integer'}
RUN_FULL_EXTRACTION = True  # @param {type:'boolean'}
NUM_SHARDS = 1  # @param {type:'integer'}
SHARD_INDEX = 0  # @param {type:'integer'}
RUN_GRAPH_PREPARATION = True  # @param {type:'boolean'}
OVERWRITE = False  # @param {type:'boolean'}

if CLASS_COUNT == 1 or CLASS_COUNT < 0:
    raise ValueError('CLASS_COUNT phải là 0 (tất cả) hoặc từ 2 trở lên.')
if VIEW_MODE not in {'three_view', 'center'}:
    raise ValueError("VIEW_MODE phải là 'three_view' hoặc 'center'.")
if VIDEO_SOURCE not in {'drive_api', 'local_dir'}:
    raise ValueError("VIDEO_SOURCE phải là 'drive_api' hoặc 'local_dir'.")
if PILOT_LIMIT < 1 or DOWNLOAD_WORKERS < 1:
    raise ValueError('PILOT_LIMIT và DOWNLOAD_WORKERS phải lớn hơn 0.')
if NUM_SHARDS < 1 or not 0 <= SHARD_INDEX < NUM_SHARDS:
    raise ValueError('Cần 0 <= SHARD_INDEX < NUM_SHARDS.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

SUBSET_NAME = 'all_classes' if CLASS_COUNT == 0 else f'top{CLASS_COUNT}'
PROJECT_ROOT = Path('/content/silent-signal')
RUNTIME_ROOT = Path('/content/multi_vsl_runtime')
METADATA_REPO = RUNTIME_ROOT / 'Multi-VSL_WACV_2025'
METADATA_ROOT = METADATA_REPO / 'data' / 'label_1_200'
VIDEO_ROOT = RUNTIME_ROOT / 'videos'

RESULTS_ROOT = Path('/content/drive/MyDrive/silent-signal-results/multi-vsl') / f'mvsl200_{VIEW_MODE}_{SUBSET_NAME}_pose'
PREPARED_ROOT = RESULTS_ROOT / 'prepared'
REQUIRED_LIST = PREPARED_ROOT / 'required_videos.txt'
MANIFEST = PREPARED_ROOT / 'manifest.csv'
SELECTION = PREPARED_ROOT / 'selection.json'
POSE_ROOT = RESULTS_ROOT / 'pose/rtmpose_l_coco_wholebody_384x288'
POSE_OUTPUT_ROOT = POSE_ROOT / 'raw'
PINNED_CONFIG = POSE_ROOT / 'provenance/rtmpose-colab-pinned.yaml'
GRAPH_ROOT = RESULTS_ROOT / 'graph/coco_wholebody_75_v1_t64'

POSE_ENV_ROOT = Path('/content/pose-env')
MMPOSE_ROOT = Path('/content/mmpose-v1.3.2')
MODEL_ROOT = Path('/content/drive/MyDrive/silent-signal-models/openmmlab')
POSE_CHECKPOINT = MODEL_ROOT / 'rtmpose-l-wholebody-384x288.pth'
DET_CHECKPOINT = MODEL_ROOT / 'rtmdet-m-person.pth'

for path in (RUNTIME_ROOT, VIDEO_ROOT, PREPARED_ROOT, POSE_OUTPUT_ROOT, PINNED_CONFIG.parent, GRAPH_ROOT, MODEL_ROOT):
    path.mkdir(parents=True, exist_ok=True)
print('Video tạm thời:', VIDEO_ROOT)
print('Kết quả lưu trên Drive:', RESULTS_ROOT)

## Lấy mã nguồn và metadata chính thức

Clone đúng nhánh `PROJECT_GIT_REF` của dự án và repo metadata của tác giả Multi-VSL. Nếu nhánh chưa được push lên GitHub, cell sẽ dừng.

In [ ]:
import os
import subprocess
import sys
import time

def run(command, cwd=None, env=None):
    command = [str(part) for part in command]
    print('+', ' '.join(command), flush=True)
    started = time.perf_counter()
    process = subprocess.Popen(
        command, cwd=cwd, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    try:
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
    print(f'[{(time.perf_counter() - started) / 60:.1f} min]', flush=True)
    if code:
        raise subprocess.CalledProcessError(code, command)

if not PROJECT_ROOT.exists():
    run(['git', 'clone', '--depth', '1', '--branch', PROJECT_GIT_REF, PROJECT_GIT_URL, PROJECT_ROOT])
else:
    run(['git', '-C', PROJECT_ROOT, 'fetch', '--depth', '1', 'origin', PROJECT_GIT_REF])
    run(['git', '-C', PROJECT_ROOT, 'checkout', '--detach', 'FETCH_HEAD'])
if not METADATA_REPO.exists():
    run(['git', 'clone', '--depth', '1', OFFICIAL_REPOSITORY, METADATA_REPO])

PROJECT_COMMIT = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
METADATA_COMMIT = subprocess.check_output(['git', '-C', str(METADATA_REPO), 'rev-parse', 'HEAD'], text=True).strip()
required_files = [
    PROJECT_ROOT / 'src/silent_signal/cli/prepare_multi_vsl_pose.py',
    PROJECT_ROOT / 'configs/pose/rtmpose.yaml',
    PROJECT_ROOT / 'configs/preprocessing/coco_wholebody_75_t64.yaml',
    METADATA_ROOT / 'train_1_200_center_ord1.csv',
    METADATA_ROOT / 'train_1_200_three_view_ord1.csv',
]
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise RuntimeError('Thiếu file cần thiết: ' + ', '.join(missing))
print('Project commit:', PROJECT_COMMIT)
print('Metadata commit:', METADATA_COMMIT)

## Môi trường RTMPose cố định

Tạo Python 3.11 riêng trong `/content/pose-env` với bộ phiên bản đã kiểm chứng trên Colab: NumPy 1.26.4, PyTorch 2.1.0 + CUDA 12.1, MMCV 2.1.0, MMDetection 3.2.0, MMPose 1.3.2. Kernel Colab không import các thư viện này nên không bị xung đột ABI. Nếu marker môi trường còn trong runtime hiện tại, phần cài nặng được bỏ qua.

In [ ]:
import shutil

run(['nvidia-smi'])
print(f"/content còn trống: {shutil.disk_usage('/content').free / 1024**3:.1f} GiB")
if shutil.which('ffprobe') is None:
    raise RuntimeError('Không tìm thấy ffprobe; cài ffmpeg hoặc đặt VALIDATION_LEVEL=metadata.')
run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv'])
run(['uv', 'python', 'install', '3.11'])
if not (POSE_ENV_ROOT / 'bin/python').is_file():
    run(['uv', 'venv', POSE_ENV_ROOT, '--python', '3.11', '--seed'])

POSE_PY = str(POSE_ENV_ROOT / 'bin/python')
POSE_CLI = str(POSE_ENV_ROOT / 'bin/ss-extract-pose')
MULTI_VSL_CLI = str(POSE_ENV_ROOT / 'bin/ss-prepare-multi-vsl-pose')
GRAPH_CLI = str(POSE_ENV_ROOT / 'bin/ss-prepare-pose-graph')
ENV_MARKER = POSE_ENV_ROOT / '.silent-signal-rtmpose-v1.ready'

if not MMPOSE_ROOT.exists():
    run(['git', 'clone', '--depth', '1', '--branch', 'v1.3.2', 'https://github.com/open-mmlab/mmpose.git', MMPOSE_ROOT])

if not ENV_MARKER.is_file():
    run([POSE_PY, '-m', 'pip', 'install', 'pip==24.3.1', 'setuptools==75.6.0', 'wheel==0.45.1'])
    run([POSE_PY, '-m', 'pip', 'install', 'numpy==1.26.4'])
    run([POSE_PY, '-m', 'pip', 'install', '--no-build-isolation', 'chumpy==0.70'])
    run([POSE_PY, '-m', 'pip', 'install', 'torch==2.1.0', 'torchvision==0.16.0',
         '--index-url', 'https://download.pytorch.org/whl/cu121',
         '--extra-index-url', 'https://pypi.org/simple'])
    run([POSE_PY, '-m', 'pip', 'install', 'mmengine==0.10.7'])
    run([POSE_PY, '-m', 'pip', 'install', 'mmcv==2.1.0',
         '-f', 'https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html'])
    run([POSE_PY, '-m', 'pip', 'install', 'mmdet==3.2.0'])
    run([POSE_PY, '-m', 'pip', 'install', '-e', MMPOSE_ROOT])

run([POSE_PY, '-m', 'pip', 'install', '-e', PROJECT_ROOT])
environment_check = '''
import mmcv, mmdet, mmpose, numpy as np, torch
from mmcv.ops import nms
assert torch.cuda.is_available(), 'CUDA không khả dụng trong pose-env.'
print('GPU:', torch.cuda.get_device_name(0))
print('NumPy', np.__version__, '| torch', torch.__version__, '| MMCV', mmcv.__version__, '| MMPose', mmpose.__version__)
'''
run([POSE_PY, '-c', environment_check])
ENV_MARKER.write_text(PROJECT_COMMIT + '\n', encoding='utf-8')

PROCESS_ENV = os.environ.copy()
PROCESS_ENV.update({
    'PYTHONUNBUFFERED': '1',
    'MPLBACKEND': 'Agg',
    'MMPOSE_ROOT': str(MMPOSE_ROOT),
    'RTMPOSE_L_WHOLEBODY_CHECKPOINT': str(POSE_CHECKPOINT),
    'RTMDET_M_PERSON_CHECKPOINT': str(DET_CHECKPOINT),
})

## Chọn lớp và danh sách video cần dùng

Lớp được xếp theo **số clip center trong tập train chính thức** (validation/test chỉ dùng để yêu cầu lớp có mặt ở cả 3 split). Khi số clip bằng nhau (148/199 lớp cùng có 22 clip train), thứ tự được quyết định bởi nhãn gốc tăng dần, nên kết quả luôn tất định. `class_index` mới chạy từ 0 theo đúng thứ hạng này, khớp với thứ tự lớp của baseline RGB. Với `three_view`, danh sách video gồm cả 3 góc của mọi lần ký thuộc các lớp đã chọn.

In [ ]:
import json

run([MULTI_VSL_CLI, 'list-videos', '--metadata-root', METADATA_ROOT, '--classes', CLASS_COUNT,
     '--views', VIEW_MODE, '--output', REQUIRED_LIST], env=PROCESS_ENV)
REQUIRED_VIDEOS = [line for line in REQUIRED_LIST.read_text(encoding='utf-8').splitlines() if line]
view_counts = {view: sum(f'_{view}_' in name for name in REQUIRED_VIDEOS) for view in ('center', 'left', 'right')}
print(f'Cần {len(REQUIRED_VIDEOS):,} video: {view_counts}')

## Tải video vào runtime

Video đã có trong `VIDEO_ROOT` với đúng kích thước sẽ không tải lại. File được ghi vào `.part` trước rồi mới đổi tên, nên một lần tải bị ngắt không để lại file hỏng.

In [ ]:
import concurrent.futures
import hashlib
import io
import threading

def local_ok(name, size=None):
    path = VIDEO_ROOT / name
    return path.is_file() and path.stat().st_size > 0 and (size is None or path.stat().st_size == int(size))

todo = [name for name in REQUIRED_VIDEOS if not local_ok(name)]
print(f'Đã có {len(REQUIRED_VIDEOS) - len(todo):,}/{len(REQUIRED_VIDEOS):,}; cần lấy {len(todo):,}.')
unavailable = []

if todo and VIDEO_SOURCE == 'local_dir':
    source_dir = Path(LOCAL_SOURCE_DIR)
    if not source_dir.is_dir():
        raise FileNotFoundError(f'LOCAL_SOURCE_DIR không tồn tại: {source_dir}')
    for position, name in enumerate(todo, start=1):
        source = source_dir / name
        if not source.is_file():
            unavailable.append(name)
            continue
        partial = VIDEO_ROOT / f'.{name}.part'
        shutil.copyfile(source, partial)
        partial.replace(VIDEO_ROOT / name)
        if position % 100 == 0 or position == len(todo):
            print(f'[copy] {position}/{len(todo)}', flush=True)

elif todo and VIDEO_SOURCE == 'drive_api':
    from google.colab import auth
    auth.authenticate_user()
    import google.auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload

    credentials, _ = google.auth.default()

    def drive_service():
        return build('drive', 'v3', credentials=credentials, cache_discovery=False)

    service = drive_service()
    wanted = set(todo)
    remote = {}
    folders = [OFFICIAL_DRIVE_FOLDER_ID]
    listed = 0
    while folders:
        folder_id = folders.pop()
        page_token = None
        while True:
            response = service.files().list(
                q=f"'{folder_id}' in parents and trashed = false",
                fields='nextPageToken, files(id, name, mimeType, size, md5Checksum)',
                pageSize=1000, pageToken=page_token,
                supportsAllDrives=True, includeItemsFromAllDrives=True,
            ).execute()
            for item in response.get('files', []):
                listed += 1
                if item['mimeType'] == 'application/vnd.google-apps.folder':
                    folders.append(item['id'])
                elif item['name'] in wanted:
                    if item['name'] in remote:
                        raise RuntimeError(f"Tên video trùng trên Drive: {item['name']}")
                    remote[item['name']] = item
            page_token = response.get('nextPageToken')
            if not page_token:
                break
        print(f'[list] đã duyệt {listed:,} mục; khớp {len(remote):,}/{len(wanted):,}', flush=True)
    if listed == 0:
        raise RuntimeError(
            'Drive API không liệt kê được thư mục của tác giả. Mở link thư mục trong trình duyệt '
            "bằng cùng tài khoản, hoặc Add shortcut to Drive rồi dùng VIDEO_SOURCE='local_dir'."
        )
    unavailable = sorted(wanted - set(remote))

    local = threading.local()

    def download(item):
        if local_ok(item['name'], item.get('size')):
            return item['name']
        if not hasattr(local, 'service'):
            local.service = drive_service()
        partial = VIDEO_ROOT / f".{item['name']}.part"
        digest = hashlib.md5()
        with partial.open('wb') as handle:
            downloader = MediaIoBaseDownload(handle, local.service.files().get_media(fileId=item['id'], supportsAllDrives=True))
            done = False
            while not done:
                _, done = downloader.next_chunk(num_retries=5)
        with partial.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                digest.update(chunk)
        if item.get('md5Checksum') and digest.hexdigest() != item['md5Checksum']:
            partial.unlink(missing_ok=True)
            raise RuntimeError(f"MD5 không khớp: {item['name']}")
        partial.replace(VIDEO_ROOT / item['name'])
        return item['name']

    items = [remote[name] for name in sorted(remote)]
    failures = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
        futures = {pool.submit(download, item): item['name'] for item in items}
        for position, future in enumerate(concurrent.futures.as_completed(futures), start=1):
            try:
                future.result()
            except Exception as exc:
                failures.append((futures[future], str(exc)))
            if position % 100 == 0 or position == len(items):
                print(f'[download] {position}/{len(items)}; lỗi {len(failures)}', flush=True)
    if failures:
        print(json.dumps(failures[:20], ensure_ascii=False, indent=2))
        raise RuntimeError(f'{len(failures)} video tải lỗi; chạy lại cell này để thử tiếp.')

if unavailable:
    (PREPARED_ROOT / 'unavailable_videos.txt').write_text('\n'.join(unavailable) + '\n', encoding='utf-8')
    raise FileNotFoundError(
        f'{len(unavailable)} video chính thức không có ở nguồn; ví dụ {unavailable[0]}. '
        f"Danh sách: {PREPARED_ROOT / 'unavailable_videos.txt'}. "
        'Thư mục công khai của tác giả chỉ có 1.000 video mẫu; hãy xin bộ đầy đủ rồi dùng '
        "VIDEO_SOURCE='local_dir' hoặc đổi OFFICIAL_DRIVE_FOLDER_ID."
    )
missing = [name for name in REQUIRED_VIDEOS if not local_ok(name)]
if missing:
    raise FileNotFoundError(f'Còn thiếu {len(missing)} video trong {VIDEO_ROOT}.')
print(f'Đủ {len(REQUIRED_VIDEOS):,} video trong {VIDEO_ROOT}.')

## Kiểm tra video và ghi manifest / split

`ss-prepare-multi-vsl-pose build` kiểm tra lại signer-disjoint, không trùng sample, video tồn tại và không rỗng. Với `VALIDATION_LEVEL='probe'`, lệnh này còn đọc header từng video bằng ffprobe để lấy số frame, fps và kích thước. RTMPose sau đó từ chối cache nếu số frame hoặc kích thước giải mã được lệch so với manifest.

In [ ]:
run([MULTI_VSL_CLI, 'build', '--metadata-root', METADATA_ROOT, '--classes', CLASS_COUNT,
     '--views', VIEW_MODE, '--video-root', VIDEO_ROOT, '--output-root', PREPARED_ROOT,
     '--level', VALIDATION_LEVEL, '--workers', 8], env=PROCESS_ENV)

selection = json.loads(SELECTION.read_text(encoding='utf-8'))
split = json.loads((PREPARED_ROOT / 'split.json').read_text(encoding='utf-8'))
print(f"{selection['class_count']} lớp | góc quay: {selection['views']}")
for name in ('train', 'validation', 'test'):
    print(f"{name:<10} {split['signer_counts'][name]:>2} signer {split['instance_counts'][name]:>5} lần ký "
          f"{split['clip_counts'][name]:>5} video  signer={split['signer_ids'][name]}")
print(f"\nDanh sách {selection['class_count']} lớp (số lần ký mỗi split):")
print(f"{'rank':>4} {'class_index':>11} {'nhãn gốc':>8} {'tên':<8} {'train':>5} {'val':>4} {'test':>4}")
for item in selection['classes']:
    counts = item['instances']
    print(f"{item['rank']:>4} {item['class_index']:>11} {item['source_label']:>8} {item['gloss_name']:<8} "
          f"{counts['train']:>5} {counts['validation']:>4} {counts['test']:>4}")

## Tải model và cố định SHA-256

Checkpoint được giữ trên Drive để dùng lại giữa các lần chạy. Notebook tính SHA-256 thực tế rồi ghi vào một bản config đã pin; lúc trích xuất, extractor từ chối chạy nếu checkpoint khác bản đã pin.

In [ ]:
import yaml

POSE_URL = ('https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/'
            'rtmpose-l_simcc-coco-wholebody_pt-aic-coco_270e-384x288-eaeb96c8_20230125.pth')
DET_URL = ('https://download.openmmlab.com/mmpose/v1/projects/rtmpose/'
           'rtmdet_m_8xb32-100e_coco-obj365-person-235e8209.pth')

for url, destination in ((POSE_URL, POSE_CHECKPOINT), (DET_URL, DET_CHECKPOINT)):
    if destination.is_file():
        print('Đã có:', destination)
        continue
    partial = Path(str(destination) + '.part')
    run(['wget', '-c', url, '-O', partial])
    if partial.stat().st_size == 0:
        raise RuntimeError(f'Checkpoint tải về rỗng: {partial}')
    partial.replace(destination)

UNPINNED_LOCK = PINNED_CONFIG.parent / 'unverified-hashes.lock.json'
PINNED_LOCK = PINNED_CONFIG.parent / 'rtmpose-colab-pinned.lock.json'
SOURCE_POSE_CONFIG = PROJECT_ROOT / 'configs/pose/rtmpose.yaml'
run([POSE_CLI, 'verify', '--config', SOURCE_POSE_CONFIG, '--write-lock', UNPINNED_LOCK], env=PROCESS_ENV)
hashes = json.loads(UNPINNED_LOCK.read_text(encoding='utf-8'))
pose_config = yaml.safe_load(SOURCE_POSE_CONFIG.read_text(encoding='utf-8'))
pose_config['extractor']['pose_model']['checkpoint_sha256'] = hashes['pose_model']['checkpoint_sha256']
pose_config['extractor']['detector']['checkpoint_sha256'] = hashes['detector']['checkpoint_sha256']
PINNED_CONFIG.write_text(yaml.safe_dump(pose_config, sort_keys=False, allow_unicode=True), encoding='utf-8')
run([POSE_CLI, 'verify', '--config', PINNED_CONFIG, '--write-lock', PINNED_LOCK], env=PROCESS_ENV)
print('Pinned config:', PINNED_CONFIG)

## Pilot

Chạy thử trên `PILOT_LIMIT` clip train đầu tiên (xếp theo `sample_id`). Cache được ghi thẳng vào thư mục `raw`; lượt chạy đầy đủ sẽ kiểm tra và dùng lại các cache hợp lệ này. Nếu một cache cũ không khớp model hoặc video nguồn, CLI dừng lại chứ không âm thầm ghi đè.

In [ ]:
def run_pose(command, report_path, label):
    try:
        run(command, env=PROCESS_ENV)
    except subprocess.CalledProcessError as exc:
        # Exit code 1 means some clips failed and the report lists them; others are setup errors.
        if exc.returncode != 1 or not report_path.is_file():
            raise
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print(json.dumps({key: value for key, value in report.items() if key != 'failures'}, ensure_ascii=False, indent=2))
    if report['failed']:
        print(json.dumps(report['failures'][:20], ensure_ascii=False, indent=2))
        raise RuntimeError(f'{label}: {report["failed"]} video lỗi; xem {report_path}.')
    return report

def extract_command(report_path, *extra):
    command = [POSE_CLI, 'extract', '--config', PINNED_CONFIG, '--manifest', MANIFEST,
               '--dataset-root', VIDEO_ROOT, '--output-root', POSE_OUTPUT_ROOT,
               '--report', report_path, '--device', 'cuda:0', *extra]
    return command + (['--overwrite'] if OVERWRITE else [])

PILOT_REPORT = POSE_ROOT / f'pilot_train_{PILOT_LIMIT}.json'
if RUN_PILOT:
    started = time.perf_counter()
    run_pose(extract_command(PILOT_REPORT, '--split', 'train', '--limit', PILOT_LIMIT, '--progress-every', 1),
             PILOT_REPORT, 'Pilot')
    seconds = (time.perf_counter() - started) / PILOT_LIMIT
    print(f'~{seconds:.1f} s/clip (gồm cả nạp model) → ước tính {seconds * len(REQUIRED_VIDEOS) / NUM_SHARDS / 3600:.1f} giờ cho toàn bộ.')
else:
    print('RUN_PILOT=False — bỏ qua pilot.')

## Trích xuất toàn bộ train / validation / test

Không dùng `--split`: cả 3 split đều cần pose. Để chạy song song trên nhiều runtime Colab, mọi runtime đặt cùng `NUM_SHARDS` và mỗi runtime một `SHARD_INDEX` khác nhau; mỗi clip được gán tất định vào đúng một shard. Nếu runtime bị ngắt, chỉ cần chạy lại: các cache đã hợp lệ được bỏ qua.

In [ ]:
FULL_REPORT = POSE_ROOT / f'full_shard_{SHARD_INDEX:03d}_of_{NUM_SHARDS:03d}.json'
if RUN_FULL_EXTRACTION:
    run_pose(extract_command(FULL_REPORT, '--num-shards', NUM_SHARDS, '--shard-index', SHARD_INDEX,
                             '--continue-on-error', '--progress-every', 10),
             FULL_REPORT, 'Full extraction')
else:
    print('RUN_FULL_EXTRACTION=False — chưa chạy toàn bộ.')

## Tạo tensor đồ thị `[64, 75, 7]`

Chạy sau khi **mọi shard** đã xong. Bước này không đọc video và không cần GPU: chọn 75 khớp (13 thân, 21×2 bàn tay, 20 mặt), nội suy khoảng trống ≤ 3 frame, chuẩn hóa theo vai/hông, lấy mẫu đều 64 frame, rồi tạo 7 kênh (x, y, confidence, vận tốc x/y, vector xương x/y). Config chạy thực tế pin SHA-256 của manifest, fingerprint của extractor và số clip của từng split.

In [ ]:
if RUN_GRAPH_PREPARATION:
    full_reports = sorted(POSE_ROOT.glob(f'full_shard_*_of_{NUM_SHARDS:03d}.json'))
    if len(full_reports) != NUM_SHARDS:
        raise RuntimeError(f'Mới có {len(full_reports)}/{NUM_SHARDS} report shard; chạy đủ các shard trước.')
    fingerprints = {json.loads(path.read_text(encoding='utf-8'))['extractor_fingerprint'] for path in full_reports}
    if len(fingerprints) != 1:
        raise RuntimeError('Các shard dùng extractor khác nhau.')
    graph_config = yaml.safe_load((PROJECT_ROOT / 'configs/preprocessing/coco_wholebody_75_t64.yaml').read_text(encoding='utf-8'))
    graph_config['expected'] = {
        'manifest_sha256': hashlib.sha256(MANIFEST.read_bytes()).hexdigest(),
        'extractor_fingerprint': fingerprints.pop(),
        'clips': sum(selection['clips'].values()),
        'classes': selection['class_count'],
        'splits': selection['clips'],
    }
    GRAPH_CONFIG = GRAPH_ROOT / 'graph-config.pinned.yaml'
    GRAPH_CONFIG.write_text(yaml.safe_dump(graph_config, sort_keys=False), encoding='utf-8')
    GRAPH_REPORT = GRAPH_ROOT / 'graph_preparation_report.json'
    command = [GRAPH_CLI, '--config', GRAPH_CONFIG, '--manifest', MANIFEST,
               '--pose-root', POSE_OUTPUT_ROOT, '--output-root', GRAPH_ROOT / 'cache',
               '--report', GRAPH_REPORT, '--progress-every', 100]
    run(command + (['--overwrite'] if OVERWRITE else []), env=PROCESS_ENV)
    graph_report = json.loads(GRAPH_REPORT.read_text(encoding='utf-8'))
    print(json.dumps({key: value for key, value in graph_report.items() if key != 'failures'}, ensure_ascii=False, indent=2))
else:
    print('RUN_GRAPH_PREPARATION=False — bỏ qua.')

## Kết quả trên Drive

`MyDrive/silent-signal-results/multi-vsl/mvsl200_<view_mode>_<subset>_pose/`

- `prepared/manifest.csv|parquet`: một dòng cho mỗi video (mỗi góc), gồm `view`, `instance_id` (chung cho 3 góc của một lần ký), `split`, `class_index`, `signer_id` và thông tin ffprobe.
- `prepared/selection.json`: **danh sách lớp** (rank, `class_index`, nhãn gốc, `VSL_NNN`, số lần ký mỗi split) cùng SHA-256 của các file CSV nguồn.
- `prepared/labels.json`, `prepared/split.json`, `prepared/validation_report.json`.
- `pose/rtmpose_l_coco_wholebody_384x288/raw/`: pose thô 133 điểm cho mỗi clip, lưu dạng NPZ không dùng pickle, có thể resume.
- `pose/.../provenance/`: config và lock chứa SHA-256 của model cùng phiên bản thư viện.
- `graph/coco_wholebody_75_v1_t64/cache/`: tensor `[64, 75, 7]` kèm mask, dùng cho graph encoder.

Video nguồn chỉ nằm trong `/content` và mất khi runtime reset. Không commit dữ liệu, manifest sinh ra hoặc cache vào Git.